# Feature Engineering with HOG

In this section, we'll implement Histogram of Oriented Gradients (HOG) feature extraction for our frames. HOG is particularly effective for object detection as it captures local gradient information.

In [ ]:
import cv2
from skimage.feature import hog
from skimage.transform import resize
import numpy as np
from tqdm import tqdm

def extract_hog_features(image_path, target_size=(224, 224)):
    """
    Extract HOG features from an image
    
    Args:
        image_path: Path to the image file
        target_size: Target size for resizing
        
    Returns:
        HOG features array
    """
    # Read and resize image
    image = cv2.imread(image_path)
    image = cv2.resize(image, target_size)
    
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Extract HOG features
    features, hog_image = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=True,
        block_norm='L2-Hys'
    )
    
    return features

def process_dataset_hog_features(frame_paths, target_size=(224, 224)):
    """
    Extract HOG features for multiple images
    
    Args:
        frame_paths: List of image file paths
        target_size: Target size for resizing
        
    Returns:
        Array of HOG features
    """
    features_list = []
    
    for path in tqdm(frame_paths, desc="Extracting HOG features"):
        features = extract_hog_features(path, target_size)
        features_list.append(features)
    
    return np.array(features_list)

# Example usage
def prepare_hog_features():
    """
    Prepare HOG features for both training and test sets
    """
    # Get paths for training data
    train_normal_paths = [os.path.join('processed_frames/train/normal', f) 
                         for f in os.listdir('processed_frames/train/normal')
                         if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    train_shoplifting_paths = [os.path.join('processed_frames/train/shoplifting', f)
                              for f in os.listdir('processed_frames/train/shoplifting')
                              if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    # Get paths for test data
    test_normal_paths = [os.path.join('processed_frames/test/normal', f)
                        for f in os.listdir('processed_frames/test/normal')
                        if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    test_shoplifting_paths = [os.path.join('processed_frames/test/shoplifting', f)
                             for f in os.listdir('processed_frames/test/shoplifting')
                             if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    print("Extracting HOG features for training set...")
    train_normal_features = process_dataset_hog_features(train_normal_paths)
    train_shoplifting_features = process_dataset_hog_features(train_shoplifting_paths)
    
    print("\nExtracting HOG features for test set...")
    test_normal_features = process_dataset_hog_features(test_normal_paths)
    test_shoplifting_features = process_dataset_hog_features(test_shoplifting_paths)
    
    # Create labels
    train_labels = np.concatenate([
        np.zeros(len(train_normal_features)),  # 0 for normal
        np.ones(len(train_shoplifting_features))  # 1 for shoplifting
    ])
    
    test_labels = np.concatenate([
        np.zeros(len(test_normal_features)),  # 0 for normal
        np.ones(len(test_shoplifting_features))  # 1 for shoplifting
    ])
    
    # Combine features
    X_train = np.vstack([train_normal_features, train_shoplifting_features])
    X_test = np.vstack([test_normal_features, test_shoplifting_features])
    
    return X_train, X_test, train_labels, test_labels

# Extract features
print("Starting HOG feature extraction...")
X_train, X_test, y_train, y_test = prepare_hog_features()

print("\nFeature extraction complete!")
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

# Model Setup - EfficientNetV2B0

Now we'll set up our model using EfficientNetV2B0 as the base model with frozen layers and add our custom classification layers on top.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

def create_model(input_shape=(224, 224, 3)):
    """
    Create and compile the model using EfficientNetV2B0 as base
    
    Args:
        input_shape: Input shape for the model
        
    Returns:
        Compiled model
    """
    # Load the base model with pre-trained weights
    base_model = EfficientNetV2B0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Freeze the base model layers
    for layer in base_model.layers:
        layer.trainable = False
    
    # Add custom layers on top
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    # Create the final model
    model = Model(inputs=base_model.input, outputs=outputs)
    
    # Compile the model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )
    
    return model

# Create the model
model = create_model()
model.summary()

# Data Augmentation

Set up data augmentation pipeline to increase dataset diversity and improve model generalization.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create data generators with augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values
    rotation_range=20,  # Random rotation
    width_shift_range=0.2,  # Random horizontal shift
    height_shift_range=0.2,  # Random vertical shift
    horizontal_flip=True,  # Random horizontal flip
    zoom_range=0.2,  # Random zoom
    fill_mode='nearest'  # Strategy for filling new pixels
)

# Create data generator for validation (only rescaling)
test_datagen = ImageDataGenerator(rescale=1./255)

# Set up the generators
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    'processed_frames/train',
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='binary',
    classes=['normal', 'shoplifting']
)

test_generator = test_datagen.flow_from_directory(
    'processed_frames/test',
    target_size=(224, 224),
    batch_size=batch_size,
    class_mode='binary',
    classes=['normal', 'shoplifting']
)

print("Data generators created successfully!")
print(f"Training samples: {train_generator.samples}")
print(f"Testing samples: {test_generator.samples}")
print(f"Class mapping: {train_generator.class_indices}")

# Model Training

Train the model using our prepared data generators with proper callbacks for monitoring and saving the best model.

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt

# Set up callbacks
callbacks = [
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
epochs = 50
steps_per_epoch = train_generator.samples // batch_size
validation_steps = test_generator.samples // batch_size

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_data=test_generator,
    validation_steps=validation_steps,
    callbacks=callbacks
)

# Plot training history
plt.figure(figsize=(12, 4))

# Plot training & validation accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

# Model Evaluation

Evaluate the model's performance using various metrics including accuracy, precision, recall, F1 score, and confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

def evaluate_model(model, test_generator):
    """
    Evaluate the model and print various metrics
    """
    # Get predictions
    test_generator.reset()
    y_pred = model.predict(test_generator, steps=len(test_generator))
    y_pred_classes = (y_pred > 0.5).astype(int)
    
    # Get true labels
    y_true = test_generator.classes
    
    # Print classification report
    print("\nClassification Report:")
    print("--------------------")
    print(classification_report(y_true, y_pred_classes, target_names=['Normal', 'Shoplifting']))
    
    # Calculate and plot confusion matrix
    cm = confusion_matrix(y_true, y_pred_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Shoplifting'],
                yticklabels=['Normal', 'Shoplifting'])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # Print additional metrics
    test_loss, test_acc, test_precision, test_recall = model.evaluate(test_generator)
    f1_score = 2 * (test_precision * test_recall) / (test_precision + test_recall)
    
    print("\nFinal Metrics:")
    print("-------------")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall: {test_recall:.4f}")
    print(f"Test F1 Score: {f1_score:.4f}")

# Evaluate the model
print("Evaluating model performance...")
evaluate_model(model, test_generator)

In [ ]:
import pandas as pd

# Feature Extraction Process

This notebook implements deep feature extraction from surveillance video frames using the following steps:

1. **Dataset Organization**
   - Organizes videos from DCSASS dataset into shoplifting and non-shoplifting categories
   - Separates data into training and testing sets

2. **Frame Extraction**
   - Extracts frames from videos at regular intervals
   - Processes both shoplifting and normal videos
   - Saves frames in organized directory structure

3. **Feature Extraction**
   - Uses pre-trained VGG16 model
   - Processes frames in memory-efficient batches
   - Applies feature standardization
   - Saves extracted features as NumPy arrays

4. **Output**
   - `train_features.npy`: Features from training set frames
   - `train_labels.npy`: Labels for training set
   - `test_features.npy`: Features from testing set frames
   - `test_labels.npy`: Labels for testing set

The features can then be used for training a shoplifting detection model.

In [ ]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.preprocessing import StandardScaler

In [ ]:
import cv2
import os

In [10]:
import zipfile
import os

def unzip_file(zip_file, extract_to):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

# Replace 'folder_archiv.zip' with the actual name of your ZIP file
zip_file = 'archive.zip'

# Replace 'extracted_folder' with the directory where you want to extract the contents
extract_to = 'extracted_folder'

# Create the extraction directory if it doesn't exist
if not os.path.exists(extract_to):
    os.makedirs(extract_to)

# Call the function to unzip the file
unzip_file(zip_file, extract_to)

print("Extraction complete!")


Extraction complete!


In [11]:
import pandas as pd
import os

def create_dataset_csv(dataset_path, output_csv="data/shoplifting.csv"):
    """
    Create a CSV file that organizes the DCSASS dataset for training
    """
    # Find all video files
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv']
    video_files = []
    
    # Create data directory if it doesn't exist
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    
    # Walk through the dataset directory to find all video files
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if any(file.endswith(ext) for ext in video_extensions):
                video_files.append(os.path.join(root, file))
    
    if not video_files:
        print(f"No video files found in {dataset_path}")
        return None
    
    # Create DataFrame - determine labels based on filenames/folder structure
    data = []
    for video_path in video_files:
        video_name = os.path.basename(video_path)
        folder_name = os.path.basename(os.path.dirname(video_path))
        
        # Determine label based on filename or folder structure
        # This is a critical step - you need to figure out how to identify shoplifting videos
        if ("shoplifting" in video_name.lower() or 
            "theft" in video_name.lower() or 
            "shoplifting" in folder_name.lower() or
            "theft" in folder_name.lower()):
            label = "shoplifting"
        else:
            label = "normal"  # Or whatever the negative class is called
        
        data.append({
            "video_path": video_path,
            "video_name": video_name,
            "folder": folder_name,
            "label": label
        })
    
    # Create DataFrame and save as CSV
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Created dataset CSV at {output_csv} with {len(df)} entries")
    print(f"Class distribution:")
    print(df["label"].value_counts())
    
    # Show some examples
    print("\nSample of created CSV:")
    print(df.head(10))
    
    return df

# Create the CSV file
dataset_path = "extracted_folder/DCSASS"  # Your dataset path
df = create_dataset_csv(dataset_path)

# If the CSV was created successfully, let's examine it
if df is not None:
    print("\n" + "="*50)
    print("DATASET ANALYSIS")
    print("="*50)
    
    # Check if we need to manually correct some labels
    print("\nChecking if manual label adjustment is needed:")
    
    # Show unique folder names
    print("\nUnique folders in dataset:")
    for folder in df['folder'].unique():
        print(f"  - {folder}")
    
    # Show unique video names (first 10)
    print("\nSample video names:")
    for video in df['video_name'].head(10):
        print(f"  - {video}")
    
    # Check if we have a good balance of classes
    shoplifting_count = len(df[df['label'] == 'shoplifting'])
    normal_count = len(df[df['label'] == 'normal'])
    
    print(f"\nClass balance:")
    print(f"Shoplifting videos: {shoplifting_count}")
    print(f"Normal videos: {normal_count}")
    
    if shoplifting_count == 0:
        print("\nWARNING: No shoplifting videos detected!")
        print("You may need to manually inspect the dataset structure and update the labeling logic.")
        print("Common approaches:")
        print("1. Check if shoplifting videos are in specific folders")
        print("2. Look for pattern in filenames (e.g., 'theft', 'steal', 'shoplift')")
        print("3. Manually review some videos to understand the dataset structure")
    
    # Save a sample of the dataset for manual inspection
    sample_df = df.head(20)[['video_name', 'folder', 'label']]
    sample_df.to_csv('data/dataset_sample_for_review.csv', index=False)
    print(f"\nSaved sample for manual review to: data/dataset_sample_for_review.csv")

Created dataset CSV at data/shoplifting.csv with 16639 entries
Class distribution:
label
normal         15743
shoplifting      896
Name: count, dtype: int64

Sample of created CSV:
                                          video_path            video_name  \
0  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   Abuse001_x264_0.mp4   
1  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   Abuse001_x264_1.mp4   
2  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_10.mp4   
3  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_11.mp4   
4  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_12.mp4   
5  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_13.mp4   
6  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_14.mp4   
7  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_15.mp4   
8  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_16.mp4   
9  extracted_folder/DCSASS\Abuse\Abuse0

In [12]:
# After running the above code, verify the CSV was created
import pandas as pd

try:
    df = pd.read_csv("data/shoplifting.csv")
    print("CSV file loaded successfully!")
    print(f"Shape: {df.shape}")
    print("\nFirst few rows:")
    print(df.head())
    
    # Now you can use this CSV file with the GitHub code
    print("\nYou can now use: df = pd.read_csv('data/shoplifting.csv')")
    
except FileNotFoundError:
    print("CSV file not found. There might have been an issue creating it.")

CSV file loaded successfully!
Shape: (16639, 4)

First few rows:
                                          video_path            video_name  \
0  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   Abuse001_x264_0.mp4   
1  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   Abuse001_x264_1.mp4   
2  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_10.mp4   
3  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_11.mp4   
4  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_12.mp4   

              folder   label  
0  Abuse001_x264.mp4  normal  
1  Abuse001_x264.mp4  normal  
2  Abuse001_x264.mp4  normal  
3  Abuse001_x264.mp4  normal  
4  Abuse001_x264.mp4  normal  

You can now use: df = pd.read_csv('data/shoplifting.csv')


In [13]:
import pandas as pd
import os

# Load the created CSV
df = pd.read_csv("data/shoplifting.csv")

# Let's see how the labeling was done
print("Unique folders in the dataset:")
print(df['folder'].unique())

print("\nHow shoplifting videos were identified:")
shoplifting_df = df[df['label'] == 'shoplifting']
print("Shoplifting folders:")
print(shoplifting_df['folder'].unique())

print("\nSample shoplifting video names:")
print(shoplifting_df['video_name'].head(20).tolist())

Unique folders in the dataset:
['Abuse001_x264.mp4' 'Abuse003_x264.mp4' 'Abuse004_x264.mp4'
 'Abuse006_x264.mp4' 'Abuse007_x264.mp4' 'Abuse008_x264.mp4'
 'Abuse010_x264.mp4' 'Abuse012_x264.mp4' 'Abuse013_x264.mp4'
 'Abuse014_x264.mp4' 'Abuse016_x264.mp4' 'Abuse017_x264.mp4'
 'Abuse018_x264.mp4' 'Abuse019_x264.mp4' 'Abuse020_x264.mp4'
 'Abuse021_x264.mp4' 'Abuse022_x264.mp4' 'Abuse023_x264.mp4'
 'Abuse024_x264.mp4' 'Abuse025_x264.mp4' 'Abuse026_x264.mp4'
 'Abuse027_x264.mp4' 'Abuse028_x264.mp4' 'Abuse030_x264.mp4'
 'Abuse031_x264.mp4' 'Abuse033_x264.mp4' 'Abuse034_x264.mp4'
 'Abuse035_x264.mp4' 'Abuse036_x264.mp4' 'Abuse037_x264.mp4'
 'Abuse039_x264.mp4' 'Abuse041_x264.mp4' 'Abuse042_x264.mp4'
 'Abuse043_x264.mp4' 'Abuse044_x264.mp4' 'Abuse045_x264.mp4'
 'Abuse047_x264.mp4' 'Abuse048_x264.mp4' 'Abuse050_x264.mp4'
 'Arrest001_x264.mp4' 'Arrest002_x264.mp4' 'Arrest003_x264.mp4'
 'Arrest004_x264.mp4' 'Arrest005_x264.mp4' 'Arrest006_x264.mp4'
 'Arrest009_x264.mp4' 'Arrest011_x264.mp4' 'Arre

In [14]:
import os

def find_data_files(directory, extensions):
    """Find files with specific extensions in a directory"""
    data_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if any(file.endswith(ext) for ext in extensions):
                data_files.append(os.path.join(root, file))
    return data_files

# Search for various data file types
data_extensions = ['.csv', '.txt', '.json', '.xml', '.xlsx']
data_files = find_data_files("extracted/DCSASS", data_extensions)

if data_files:
    print("Found these data files:")
    for file in data_files:
        print(f"  - {file}")
        
    # If we found CSV files, let's examine them
    csv_files = [f for f in data_files if f.endswith('.csv')]
    if csv_files:
        print("\nLet's examine the CSV files:")
        import pandas as pd
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                print(f"\nContents of {csv_file}:")
                print(f"Shape: {df.shape}")
                print("Columns:", df.columns.tolist())
                print("\nFirst few rows:")
                print(df.head())
            except Exception as e:
                print(f"Error reading {csv_file}: {e}")
else:
    print("No data files found. Let's check what video files are available:")
    
    # Look for video files instead
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.wmv']
    video_files = find_data_files("extracted_folder/DCSASS", video_extensions)
    
    if video_files:
        print(f"Found {len(video_files)} video files:")
        for video_file in video_files[:5]:  # Show first 5
            print(f"  - {video_file}")
        if len(video_files) > 5:
            print(f"  ... and {len(video_files) - 5} more")
        
        print("\nYou might need to create your own CSV file that references these videos.")
    else:
        print("No video files found either. Please check if the dataset was extracted correctly.")

No data files found. Let's check what video files are available:
Found 16639 video files:
  - extracted_folder/DCSASS\Abuse\Abuse001_x264.mp4\Abuse001_x264_0.mp4
  - extracted_folder/DCSASS\Abuse\Abuse001_x264.mp4\Abuse001_x264_1.mp4
  - extracted_folder/DCSASS\Abuse\Abuse001_x264.mp4\Abuse001_x264_10.mp4
  - extracted_folder/DCSASS\Abuse\Abuse001_x264.mp4\Abuse001_x264_11.mp4
  - extracted_folder/DCSASS\Abuse\Abuse001_x264.mp4\Abuse001_x264_12.mp4
  ... and 16634 more

You might need to create your own CSV file that references these videos.


In [15]:
# Now you can use your CSV for the next steps in the GitHub code
df = pd.read_csv("data/shoplifting.csv")

print("Dataset ready for processing!")
print(f"Total videos: {len(df)}")
print(f"Shoplifting: {len(df[df['label'] == 'shoplifting'])}")
print(f"Normal: {len(df[df['label'] == 'normal'])}")

# The GitHub code should now work with:
# df = pd.read_csv("data/shoplifting.csv")

Dataset ready for processing!
Total videos: 16639
Shoplifting: 896
Normal: 15743


In [ ]:
# Load the GitHub-style dataset
df = pd.read_csv("data/github_format_dataset.csv", index_col=0)

print("Before dropping Shoplifting column:")
print(df.head())
print(f"Shape: {df.shape}")

# This is exactly what the GitHub code does:
df.drop("Shoplifting", axis=1, inplace=True)

print("\nAfter dropping Shoplifting column:")
print(df.head())
print(f"Shape: {df.shape}")

# Now you have just the index (video names) as your features
# The GitHub code probably continues with feature extraction from these video names

Before dropping Shoplifting column:
                            Shoplifting
video_name                             
Shoplifting001_x264_0.mp4             1
Shoplifting001_x264_1.mp4             1
Shoplifting001_x264_10.mp4            1
Shoplifting001_x264_11.mp4            1
Shoplifting001_x264_12.mp4            1
Shape: (3584, 1)

After dropping Shoplifting column:
Empty DataFrame
Columns: []
Index: [Shoplifting001_x264_0.mp4, Shoplifting001_x264_1.mp4, Shoplifting001_x264_10.mp4, Shoplifting001_x264_11.mp4, Shoplifting001_x264_12.mp4]
Shape: (3584, 0)


In [ ]:
df.head()

""
video_name
Shoplifting001_x264_0.mp4
Shoplifting001_x264_1.mp4
Shoplifting001_x264_10.mp4
Shoplifting001_x264_11.mp4
Shoplifting001_x264_12.mp4


In [ ]:
import pandas as pd
import numpy as np

# Load your original dataset
df = pd.read_csv("data/shoplifting.csv")

print("Original dataset:")
print(f"Total videos: {len(df)}")
print("Current label distribution:")
print(df['label'].value_counts())

# Let's analyze the folder names to create proper categories
print("\nUnique folder names:")
for folder in sorted(df['folder'].unique()):
    print(f"  - {folder}")

# Create proper labels based on folder names
def categorize_activity(folder_name):
    folder_lower = folder_name.lower()
    
    if 'shoplift' in folder_lower or 'steal' in folder_lower:
        return 'shoplifting'
    elif 'abuse' in folder_lower:
        return 'abuse'
    elif 'arrest' in folder_lower or 'arson' in folder_lower or 'assault' in folder_lower:
        return 'suspicious'
    elif 'normal' in folder_lower or 'shopping' in folder_lower or 'customer' in folder_lower:
        return 'normal'
    else:
        return 'other'

# Apply the categorization
df['activity_type'] = df['folder'].apply(categorize_activity)

print("\nNew activity type distribution:")
print(df['activity_type'].value_counts())

# Let's see some examples
print("\nSample videos with new labels:")
sample_df = df[['video_name', 'folder', 'activity_type']].sample(10)
print(sample_df)

Original dataset:
Total videos: 33278
Current label distribution:
label
normal         31486
shoplifting     1792
Name: count, dtype: int64

Unique folder names:
  - Abuse001_x264.mp4
  - Abuse003_x264.mp4
  - Abuse004_x264.mp4
  - Abuse006_x264.mp4
  - Abuse007_x264.mp4
  - Abuse008_x264.mp4
  - Abuse010_x264.mp4
  - Abuse012_x264.mp4
  - Abuse013_x264.mp4
  - Abuse014_x264.mp4
  - Abuse016_x264.mp4
  - Abuse017_x264.mp4
  - Abuse018_x264.mp4
  - Abuse019_x264.mp4
  - Abuse020_x264.mp4
  - Abuse021_x264.mp4
  - Abuse022_x264.mp4
  - Abuse023_x264.mp4
  - Abuse024_x264.mp4
  - Abuse025_x264.mp4
  - Abuse026_x264.mp4
  - Abuse027_x264.mp4
  - Abuse028_x264.mp4
  - Abuse030_x264.mp4
  - Abuse031_x264.mp4
  - Abuse033_x264.mp4
  - Abuse034_x264.mp4
  - Abuse035_x264.mp4
  - Abuse036_x264.mp4
  - Abuse037_x264.mp4
  - Abuse039_x264.mp4
  - Abuse041_x264.mp4
  - Abuse042_x264.mp4
  - Abuse043_x264.mp4
  - Abuse044_x264.mp4
  - Abuse045_x264.mp4
  - Abuse047_x264.mp4
  - Abuse048_x264.mp4
  

In [ ]:
import pandas as pd
import os

# Load your dataset
df = pd.read_csv("data/shoplifting.csv")  # or your current DataFrame

print("Your current DataFrame structure:")
print(df.head())
print(f"Shape: {df.shape}")

# Transform to GitHub format
# 1. Create the full video paths in the format GitHub expects
def create_github_path(row):
    """Create path in format: data/Shoplifting/Shoplifting001_x264_0.mp4"""
    # Extract the base category (Shoplifting, Abuse, etc.)
    category = row['folder'].split('_')[0]  # Gets "Shoplifting" from "Shoplifting001_x264.mp4"
    
    # Create the path GitHub wants
    return f"data/{category}/{row['video_name']}"

# Apply the path transformation
df['github_path'] = df.apply(create_github_path, axis=1)

# 2. Create binary labels (1 for shoplifting, 0 for everything else)
df['github_label'] = (df['activity_type'] == 'shoplifting').astype(int)

# 3. Create the GitHub-style DataFrame
github_df = pd.DataFrame({
    'video_path': df['github_path'],
    'label': df['github_label']
})

print("\nTransformed to GitHub format:")
print(github_df.head())
print(f"New shape: {github_df.shape}")

# 4. Save in GitHub format
github_df.to_csv("data/github_format_dataset.csv", index=False)
print("\nSaved GitHub format dataset to: data/github_format_dataset.csv")

Your current DataFrame structure:
                                          video_path            video_name  \
0  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   Abuse001_x264_0.mp4   
1  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   Abuse001_x264_1.mp4   
2  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_10.mp4   
3  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_11.mp4   
4  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_12.mp4   

              folder   label  
0  Abuse001_x264.mp4  normal  
1  Abuse001_x264.mp4  normal  
2  Abuse001_x264.mp4  normal  
3  Abuse001_x264.mp4  normal  
4  Abuse001_x264.mp4  normal  
Shape: (33278, 4)


KeyError: 'activity_type'

In [ ]:
import pandas as pd
import os

# Load your dataset
df = pd.read_csv("data/shoplifting.csv")  # or your current DataFrame

print("Your current DataFrame structure:")
print(df.head())
print(f"Shape: {df.shape}")

# Step 1: First, let's see what columns you actually have
print("\nColumns in your DataFrame:")
print(df.columns.tolist())

# Step 2: Create the activity_type column if it doesn't exist
if 'activity_type' not in df.columns:
    print("\nCreating activity_type column...")
    def categorize_activity(folder_name):
        folder_lower = folder_name.lower()
        
        if 'shoplift' in folder_lower or 'steal' in folder_lower:
            return 'shoplifting'
        elif 'abuse' in folder_lower:
            return 'abuse'
        elif 'arrest' in folder_lower or 'arson' in folder_lower or 'assault' in folder_lower:
            return 'suspicious'
        elif 'normal' in folder_lower or 'shopping' in folder_lower or 'customer' in folder_lower:
            return 'normal'
        else:
            return 'other'
    
    df['activity_type'] = df['folder'].apply(categorize_activity)

# Step 3: Create the github_path column
print("\nCreating github_path column...")
def create_github_path(row):
    """Create path in format: data/Shoplifting_filtered/Shoplifting001_x264_0.mp4"""
    # Extract the base category (Shoplifting, Abuse, etc.)
    category = row['folder'].split('_')[0]  # Gets "Shoplifting" from "Shoplifting001_x264.mp4"
    
    # Create the path GitHub wants
    return f"data/{category}/{row['video_name']}"

df['github_path'] = df.apply(create_github_path, axis=1)

# Step 4: Create the github_label column (binary: 1 for shoplifting, 0 for everything else)
print("Creating github_label column...")
df['github_label'] = (df['activity_type'] == 'shoplifting').astype(int)

# Step 5: Now create the GitHub-style DataFrame
print("Creating GitHub format DataFrame...")
github_df = pd.DataFrame({
    'video_path': df['github_path'],
    'label': df['github_label']
})

print("\nTransformed to GitHub format:")
print(github_df.head())
print(f"New shape: {github_df.shape}")

# Step 6: Save in GitHub format
github_df.to_csv("data/github_format_dataset.csv", index=False)
print("\nSaved GitHub format dataset to: data/github_format_dataset.csv")

# Step 7: Create EXACT replica of GitHub format (no column names)
github_exact = pd.DataFrame({
    '': df['github_path'],  # Empty column name for first column
    ' ': df['github_label']  # Space as column name for second column
})

print("\nEXACT GitHub format replica:")
print(github_exact.head())

# Save without column names
github_exact.to_csv("data/github_exact_format.csv", header=False, index=False)
print("Saved exact GitHub format to: data/github_exact_format.csv")

Your current DataFrame structure:
                                          video_path            video_name  \
0  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   Abuse001_x264_0.mp4   
1  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   Abuse001_x264_1.mp4   
2  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_10.mp4   
3  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_11.mp4   
4  extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...  Abuse001_x264_12.mp4   

              folder   label  
0  Abuse001_x264.mp4  normal  
1  Abuse001_x264.mp4  normal  
2  Abuse001_x264.mp4  normal  
3  Abuse001_x264.mp4  normal  
4  Abuse001_x264.mp4  normal  
Shape: (33278, 4)

Columns in your DataFrame:
['video_path', 'video_name', 'folder', 'label']

Creating activity_type column...

Creating github_path column...
Creating github_label column...
Creating GitHub format DataFrame...

Transformed to GitHub format:
                           video_path  label
0   dat

In [ ]:
# Pivot your DataFrame to have video names as columns
pivoted_df = df.pivot(columns='video_name', values='video_path')
print("Pivoted DataFrame (video names as columns):")
print(pivoted_df.head())

# Now you could theoretically do something like:
# for col in pivoted_df.columns:
#     pivoted_df[col] = pivoted_df[col].apply(lambda x: "data/github_format_dataset/" + os.path.basename(x))

Pivoted DataFrame (video names as columns):
video_name                                Abuse001_x264_0.mp4  \
0           extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   
1                                                         NaN   
2                                                         NaN   
3                                                         NaN   
4                                                         NaN   

video_name                                Abuse001_x264_1.mp4  \
0                                                         NaN   
1           extracted_folder/DCSASS\Abuse\Abuse001_x264.mp...   
2                                                         NaN   
3                                                         NaN   
4                                                         NaN   

video_name                               Abuse001_x264_10.mp4  \
0                                                         NaN   
1                                           

In [ ]:
# Modify the video_path column to match GitHub format
df['video_path'] = df['video_path'].apply(lambda x: "data/github_format_dataset/" + os.path.basename(x))

print("Updated video paths:")
print(df[['video_name', 'video_path']].head())

Updated video paths:
             video_name                                       video_path
0   Abuse001_x264_0.mp4   data/github_format_dataset/Abuse001_x264_0.mp4
1   Abuse001_x264_1.mp4   data/github_format_dataset/Abuse001_x264_1.mp4
2  Abuse001_x264_10.mp4  data/github_format_dataset/Abuse001_x264_10.mp4
3  Abuse001_x264_11.mp4  data/github_format_dataset/Abuse001_x264_11.mp4
4  Abuse001_x264_12.mp4  data/github_format_dataset/Abuse001_x264_12.mp4


In [ ]:
df.head()

,video_path,video_name,folder,label,activity_type,github_path,github_label
0,data/github_format_dataset/Abuse001_x264_0.mp4,Abuse001_x264_0.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_0.mp4,0
1,data/github_format_dataset/Abuse001_x264_1.mp4,Abuse001_x264_1.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_1.mp4,0
2,data/github_format_dataset/Abuse001_x264_10.mp4,Abuse001_x264_10.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_10.mp4,0
3,data/github_format_dataset/Abuse001_x264_11.mp4,Abuse001_x264_11.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_11.mp4,0
4,data/github_format_dataset/Abuse001_x264_12.mp4,Abuse001_x264_12.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_12.mp4,0


In [ ]:
df = df.rename(columns={'Shoplifting001_x264_0': 'path', '0': 'target'})

In [ ]:
import os

def explore_file_structure():
    """Explore your actual file structure to understand the correct paths"""
    base_path = "extracted_folder"
    
    if not os.path.exists(base_path):
        print(f"❌ Base path '{base_path}' does not exist!")
        return False
    
    print("📁 Exploring file structure:")
    
    # Check DCSASS directory
    dcsass_path = os.path.join(base_path, "DCSASS")
    if not os.path.exists(dcsass_path):
        print(f"❌ DCSASS directory not found at: {dcsass_path}")
        print("Available directories in extracted_folder:")
        for item in os.listdir(base_path):
            print(f"  - {item}")
        return False
    
    print("✅ Found DCSASS directory")
    
    # Explore DCSASS contents
    print("\n📁 Contents of DCSASS:")
    for category in os.listdir(dcsass_path):
        category_path = os.path.join(dcsass_path, category)
        if os.path.isdir(category_path):
            print(f"  📁 {category}/")
            # Show first few items
            try:
                items = os.listdir(category_path)
                for item in items[:3]:  # Show first 3
                    item_path = os.path.join(category_path, item)
                    print(f"    - {item} ({'dir' if os.path.isdir(item_path) else 'file'})")
                if len(items) > 3:
                    print(f"    - ... and {len(items) - 3} more")
            except PermissionError:
                print("    🚫 Permission denied")
    
    return True

# Run the exploration
explore_file_structure()

📁 Exploring file structure:
✅ Found DCSASS directory

📁 Contents of DCSASS:
  📁 Abuse/
    - Abuse001_x264.mp4 (dir)
    - Abuse003_x264.mp4 (dir)
    - Abuse004_x264.mp4 (dir)
    - ... and 36 more
  📁 Arrest/
    - Arrest001_x264.mp4 (dir)
    - Arrest002_x264.mp4 (dir)
    - Arrest003_x264.mp4 (dir)
    - ... and 23 more
  📁 Arson/
    - Arson002_x264.mp4 (dir)
    - Arson003_x264.mp4 (dir)
    - Arson005_x264.mp4 (dir)
    - ... and 19 more
  📁 Assault/
    - Assault002_x264.mp4 (dir)
    - Assault004_x264.mp4 (dir)
    - Assault005_x264.mp4 (dir)
    - ... and 20 more
  📁 Burglary/
    - Burglary001_x264.mp4 (dir)
    - Burglary004_x264.mp4 (dir)
    - Burglary007_x264.mp4 (dir)
    - ... and 44 more
  📁 Explosion/
    - Explosion004_x264.mp4 (dir)
    - Explosion006_x264.mp4 (dir)
    - Explosion008_x264.mp4 (dir)
    - ... and 20 more
  📁 Fighting/
    - Fighting002_x264.mp4 (dir)
    - Fighting003_x264.mp4 (dir)
    - Fighting005_x264.mp4 (dir)
    - ... and 5 more
  📁 Labels/


True

In [ ]:
df.head()

,video_path,video_name,folder,label,activity_type,github_path,github_label
0,data/github_format_dataset/Abuse001_x264_0.mp4,Abuse001_x264_0.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_0.mp4,0
1,data/github_format_dataset/Abuse001_x264_1.mp4,Abuse001_x264_1.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_1.mp4,0
2,data/github_format_dataset/Abuse001_x264_10.mp4,Abuse001_x264_10.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_10.mp4,0
3,data/github_format_dataset/Abuse001_x264_11.mp4,Abuse001_x264_11.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_11.mp4,0
4,data/github_format_dataset/Abuse001_x264_12.mp4,Abuse001_x264_12.mp4,Abuse001_x264.mp4,normal,abuse,data/Abuse001/Abuse001_x264_12.mp4,0


In [ ]:
import os
import cv2
import pandas as pd

def extract_frames_smart(video_path, target, output_dir, video_index, max_frames_per_video=50):
    """Extract frames efficiently with limits"""
    frame_count = 0
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Warning: Could not open video {video_path}")
        return frame_count

    # Create output directory
    frame_dir = os.path.join(output_dir, "normal" if target == 0 else "abnormal")
    os.makedirs(frame_dir, exist_ok=True)

    # Get total frames and calculate sampling rate
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames > max_frames_per_video:
        frame_interval = max(1, total_frames // max_frames_per_video)
    else:
        frame_interval = 1

    # Extract frames with sampling
    current_frame = 0
    while cap.isOpened() and frame_count < max_frames_per_video:
        ret, frame = cap.read()
        if not ret:
            break
            
        if current_frame % frame_interval == 0:
            cv2.imwrite(os.path.join(frame_dir, f"video_{video_index}_frame_{frame_count}.jpg"), frame)
            frame_count += 1
            
        current_frame += 1

    cap.release()
    return frame_count

# Process in batches to avoid overwhelming the system
def process_batch(df, start_idx, end_idx, output_dir):
    """Process a batch of videos"""
    for index in range(start_idx, end_idx):
        if index >= len(df):
            break
            
        row = df.iloc[index]
        video_path = row['actual_video_path'].strip()
        target = row['label']
        
        print(f"Processing {index+1}/{len(df)}: {os.path.basename(video_path)}")
        extract_frames_smart(video_path, target, output_dir, index)

In [ ]:
df.head()

,video_path,label
0,data/Abuse001/Abuse001_x264_0.mp4,0
1,data/Abuse001/Abuse001_x264_1.mp4,0
2,data/Abuse001/Abuse001_x264_10.mp4,0
3,data/Abuse001/Abuse001_x264_11.mp4,0
4,data/Abuse001/Abuse001_x264_12.mp4,0


In [3]:
import os
import random
import shutil

# Define the paths
output_normal_dir = "output/normal"
test_normal_dir = "test/normal"

# Create the test/normal directory if it doesn't exist
if not os.path.exists(test_normal_dir):
    os.makedirs(test_normal_dir)

# Create the output/normal directory if it doesn't exist
if not os.path.exists(output_normal_dir):
    os.makedirs(output_normal_dir)

# List all files in the output/normal directory
files = os.listdir(output_normal_dir)

# Calculate the number of files to move (20%)
num_files_to_move = int(0.2 * len(files))

# Randomly select files to move
files_to_move = random.sample(files, num_files_to_move)

# Move the selected files to the test/normal directory
for file_name in files_to_move:
    src = os.path.join(output_normal_dir, file_name)
    dst = os.path.join(test_normal_dir, file_name)
    shutil.move(src, dst)

print("Images moved to the test/normal directory.")


Images moved to the test/normal directory.


# Image Feature Extraction

This section implements feature extraction for the frame images:
1. Load pre-trained VGG16 model
2. Process both normal and abnormal images
3. Save features for training and testing sets separately

In [ ]:
import os
import numpy as np
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm

def extract_image_features(img_path, model):
    """
    Extract features from a single image using VGG16
    """
    try:
        # Load and preprocess image
        img = image.load_img(img_path, target_size=(224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        
        # Extract features
        features = model.predict(x, verbose=0)
        return features.flatten()
    
    except Exception as e:
        print(f"Error processing image {img_path}: {str(e)}")
        return None

def process_directory(directory, model):
    """
    Extract features from all images in a directory
    """
    features_list = []
    file_paths = []
    
    # Get all image files
    image_files = [f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    # Process each image with progress bar
    for img_file in tqdm(image_files, desc=f"Processing {os.path.basename(directory)}"):
        img_path = os.path.join(directory, img_file)
        features = extract_image_features(img_path, model)
        
        if features is not None:
            features_list.append(features)
            file_paths.append(img_path)
    
    return np.array(features_list), file_paths

# Dataset Organization and Feature Extraction

Given the DCSASS dataset structure with categories like:
- Shoplifting (target class)
- Abuse, Arrest, Arson, Assault, etc. (non-target classes)

We'll organize the data and extract features in the following steps:
1. Organize videos into binary classes (shoplifting vs non-shoplifting)
2. Extract frames from videos
3. Apply VGG16 for feature extraction
4. Save processed features for training

In [17]:
import os

def explore_video_directories():
    """Explore what's inside the video directories"""
    base_path = "extracted_folder/DCSASS"
    
    print("🔍 Exploring video directories:")
    
    # Check a few video directories to see what's inside
    sample_categories = ['Abuse', 'Shoplifting']  # Check these categories
    
    for category in sample_categories:
        category_path = os.path.join(base_path, category)
        if os.path.exists(category_path):
            print(f"\n📁 {category}/:")
            video_dirs = os.listdir(category_path)
            
            for video_dir in video_dirs[:3]:  # Check first 3 videos
                video_dir_path = os.path.join(category_path, video_dir)
                if os.path.isdir(video_dir_path):
                    print(f"  📁 {video_dir}/")
                    # Show contents of this video directory
                    try:
                        contents = os.listdir(video_dir_path)
                        for item in contents[:5]:  # Show first 5 items
                            print(f"    - {item}")
                        if len(contents) > 5:
                            print(f"    - ... and {len(contents) - 5} more")
                    except PermissionError:
                        print("    🚫 Permission denied")
                else:
                    print(f"  📄 {video_dir} (file)")
        else:
            print(f"❌ Category '{category}' not found")

# Explore what's inside the video directories
explore_video_directories()

🔍 Exploring video directories:

📁 Abuse/:
  📁 Abuse001_x264.mp4/
    - Abuse001_x264_0.mp4
    - Abuse001_x264_1.mp4
    - Abuse001_x264_10.mp4
    - Abuse001_x264_11.mp4
    - Abuse001_x264_12.mp4
    - ... and 27 more
  📁 Abuse003_x264.mp4/
    - Abuse003_x264_0.mp4
    - Abuse003_x264_1.mp4
    - Abuse003_x264_10.mp4
    - Abuse003_x264_11.mp4
    - Abuse003_x264_12.mp4
    - ... and 27 more
  📁 Abuse004_x264.mp4/
    - Abuse004_x264_0.mp4
    - Abuse004_x264_1.mp4
    - Abuse004_x264_10.mp4
    - Abuse004_x264_11.mp4
    - Abuse004_x264_12.mp4
    - ... and 27 more

📁 Shoplifting/:
  📁 Shoplifting001_x264.mp4/
    - Shoplifting001_x264_0.mp4
    - Shoplifting001_x264_1.mp4
    - Shoplifting001_x264_10.mp4
    - Shoplifting001_x264_11.mp4
    - Shoplifting001_x264_12.mp4
    - ... and 27 more
  📁 Shoplifting005_x264.mp4/
    - Shoplifting005_x264_0.mp4
    - Shoplifting005_x264_1.mp4
    - Shoplifting005_x264_10.mp4
    - Shoplifting005_x264_11.mp4
    - Shoplifting005_x264_12.mp4
 

In [11]:
def organize_dataset(base_path="extracted_folder/DCSASS"):
    """
    Organize the DCSASS dataset into shoplifting and non-shoplifting categories
    """
    categories = {}
    shoplifting_videos = []
    normal_videos = []
    
    # Scan through all categories
    for category in os.listdir(base_path):
        category_path = os.path.join(base_path, category)
        if os.path.isdir(category_path):
            # Initialize category counter
            categories[category] = 0
            
            # Scan through video directories
            for video_dir in os.listdir(category_path):
                video_dir_path = os.path.join(category_path, video_dir)
                if os.path.isdir(video_dir_path):
                    # Look for videos inside each directory
                    video_files = [f for f in os.listdir(video_dir_path) 
                                 if f.endswith(('.mp4', '.avi'))]
                    
                    if video_files:
                        categories[category] += 1
                        # Get the first video file (there should be only one)
                        video_path = os.path.join(video_dir_path, video_files[0])
                        
                        # Classify video
                        if category.lower() in ['shoplifting', 'stealing', 'theft']:
                            shoplifting_videos.append(video_path)
                        else:
                            normal_videos.append(video_path)
    
    print("Dataset Statistics:")
    print("------------------")
    for category, count in categories.items():
        print(f"{category}: {count} videos")
    
    print(f"\nTotal Shoplifting videos: {len(shoplifting_videos)}")
    print(f"Total Normal videos: {len(normal_videos)}")
    
    return shoplifting_videos, normal_videos

In [12]:
def extract_frames_from_videos(video_paths, output_dir, label, frames_per_video=30):
    """
    Extract frames from a list of videos
    """
    os.makedirs(output_dir, exist_ok=True)
    
    for i, video_path in enumerate(tqdm(video_paths, desc=f"Processing {label} videos")):
        try:
            # Verify video file exists
            if not os.path.exists(video_path):
                print(f"Video file not found: {video_path}")
                continue
            
            # Try to open the video
            cap = cv2.VideoCapture(str(video_path))
            if not cap.isOpened():
                print(f"Failed to open video: {video_path}")
                print("Make sure the video file is valid and not corrupted")
                continue
            
            # Get total frames and calculate sampling interval
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames <= 0:
                print(f"No frames found in video: {video_path}")
                continue
                
            frame_interval = max(1, total_frames // frames_per_video)
            
            frame_count = 0
            current_frame = 0
            
            while cap.isOpened() and frame_count < frames_per_video:
                ret, frame = cap.read()
                if not ret:
                    break
                    
                if current_frame % frame_interval == 0:
                    # Create a unique frame name using video directory name
                    video_dir = os.path.basename(os.path.dirname(video_path))
                    frame_path = os.path.join(output_dir, 
                                            f"{label}_{video_dir}_frame_{frame_count:03d}.jpg")
                    
                    # Save frame with error checking
                    try:
                        cv2.imwrite(frame_path, frame)
                        frame_count += 1
                    except Exception as e:
                        print(f"Error saving frame from {video_path}: {str(e)}")
                
                current_frame += 1
            
            cap.release()
            
            if frame_count == 0:
                print(f"Warning: No frames were extracted from {video_path}")
            else:
                print(f"Successfully extracted {frame_count} frames from {video_path}")
            
        except Exception as e:
            print(f"Error processing video {video_path}: {str(e)}")
            continue
            
    return output_dir

In [13]:
import os
import cv2
import random
import numpy as np
from tqdm import tqdm

# 1. Organize dataset
print("Organizing dataset...")
shoplifting_videos, normal_videos = organize_dataset()

# 2. Create output directories
output_base = "processed_frames"
train_shop_dir = os.path.join(output_base, "train", "shoplifting")
train_normal_dir = os.path.join(output_base, "train", "normal")
test_shop_dir = os.path.join(output_base, "test", "shoplifting")
test_normal_dir = os.path.join(output_base, "test", "normal")

# 3. Split data into train and test
test_ratio = 0.2
random.seed(42)  # For reproducibility

test_shop = random.sample(shoplifting_videos, int(len(shoplifting_videos) * test_ratio))
train_shop = [v for v in shoplifting_videos if v not in test_shop]

test_normal = random.sample(normal_videos, int(len(normal_videos) * test_ratio))
train_normal = [v for v in normal_videos if v not in test_normal]

# 4. Extract frames
print("\nExtracting frames from videos...")
extract_frames_from_videos(train_shop, train_shop_dir, "shoplifting")
extract_frames_from_videos(train_normal, train_normal_dir, "normal")
extract_frames_from_videos(test_shop, test_shop_dir, "shoplifting")
extract_frames_from_videos(test_normal, test_normal_dir, "normal")

print("\nFrame extraction completed!")
print(f"Training Shoplifting frames: {len(os.listdir(train_shop_dir))}")
print(f"Training Normal frames: {len(os.listdir(train_normal_dir))}")
print(f"Testing Shoplifting frames: {len(os.listdir(test_shop_dir))}")
print(f"Testing Normal frames: {len(os.listdir(test_normal_dir))}")

Organizing dataset...
Dataset Statistics:
------------------
Abuse: 39 videos
Arrest: 26 videos
Arson: 22 videos
Assault: 23 videos
Burglary: 47 videos
Explosion: 23 videos
Fighting: 8 videos
Labels: 0 videos
RoadAccidents: 76 videos
Robbery: 105 videos
Shooting: 30 videos
Shoplifting: 28 videos
Stealing: 64 videos
Vandalism: 29 videos

Total Shoplifting videos: 92
Total Normal videos: 428

Extracting frames from videos...
Dataset Statistics:
------------------
Abuse: 39 videos
Arrest: 26 videos
Arson: 22 videos
Assault: 23 videos
Burglary: 47 videos
Explosion: 23 videos
Fighting: 8 videos
Labels: 0 videos
RoadAccidents: 76 videos
Robbery: 105 videos
Shooting: 30 videos
Shoplifting: 28 videos
Stealing: 64 videos
Vandalism: 29 videos

Total Shoplifting videos: 92
Total Normal videos: 428

Extracting frames from videos...


Processing shoplifting videos:   1%|▏         | 1/74 [00:00<00:08,  8.59it/s]

Successfully extracted 30 frames from extracted_folder/DCSASS\Shoplifting\Shoplifting001_x264.mp4\Shoplifting001_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Shoplifting\Shoplifting005_x264.mp4\Shoplifting005_x264_0.mp4


Processing shoplifting videos:   4%|▍         | 3/74 [00:00<00:05, 12.55it/s]

Successfully extracted 30 frames from extracted_folder/DCSASS\Shoplifting\Shoplifting006_x264.mp4\Shoplifting006_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Shoplifting\Shoplifting010_x264.mp4\Shoplifting010_x264_0.mp4


...

Successfully extracted 30 frames from extracted_folder/DCSASS\RoadAccidents\RoadAccidents109_x264.mp4\RoadAccidents109_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Burglary\Burglary055_x264.mp4\Burglary055_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Robbery\Robbery093_x264.mp4\Robbery093_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Robbery\Robbery123_x264.mp4\Robbery123_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Robbery\Robbery123_x264.mp4\Robbery123_x264_0.mp4


Processing normal videos:  95%|█████████▌| 81/85 [00:04<00:00, 17.12it/s]

Successfully extracted 30 frames from extracted_folder/DCSASS\Robbery\Robbery029_x264.mp4\Robbery029_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Robbery\Robbery121_x264.mp4\Robbery121_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Explosion\Explosion022_x264.mp4\Explosion022_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Shooting\Shooting041_x264.mp4\Shooting041_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Shooting\Shooting050_x264.mp4\Shooting050_x264_0.mp4Successfully extracted 30 frames from extracted_folder/DCSASS\Shooting\Shooting050_x264.mp4\Shooting050_x264_0.mp4

Processing normal videos: 100%|██████████| 85/85 [00:04<00:00, 17.11it/s]


Successfully extracted 30 frames from extracted_folder/DCSASS\Abuse\Abuse036_x264.mp4\Abuse036_x264_0.mp4
Successfully extracted 30 frames from extracted_folder/DCSASS\Burglary\Burglary014_x264.mp4\Burglary014_x264_0.mp4

Frame extraction completed!
Training Shoplifting frames: 2220
Training Normal frames: 10290
Testing Shoplifting frames: 540
Testing Normal frames: 2550


In [17]:
# Load the dataset from extracted frames
output_base = "processed_frames"

# Function to count frames in a directory
def count_frames_in_dir(directory):
    if not os.path.exists(directory):
        return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

# Count frames in each directory
train_shop_frames = count_frames_in_dir(os.path.join(output_base, "train", "shoplifting"))
train_normal_frames = count_frames_in_dir(os.path.join(output_base, "train", "normal"))
test_shop_frames = count_frames_in_dir(os.path.join(output_base, "test", "shoplifting"))
test_normal_frames = count_frames_in_dir(os.path.join(output_base, "test", "normal"))

print("Dataset Statistics:")
print("-----------------")
print(f"Training set:")
print(f"  - Shoplifting frames: {train_shop_frames}")
print(f"  - Normal frames: {train_normal_frames}")
print(f"\nTesting set:")
print(f"  - Shoplifting frames: {test_shop_frames}")
print(f"  - Normal frames: {test_normal_frames}")
print(f"\nTotal frames: {train_shop_frames + train_normal_frames + test_shop_frames + test_normal_frames}")

Dataset Statistics:
-----------------
Training set:
  - Shoplifting frames: 2220
  - Normal frames: 10290

Testing set:
  - Shoplifting frames: 540
  - Normal frames: 2550

Total frames: 15600


In [30]:
import pandas as pd
# Create a DataFrame with frame information
frame_data = []

# Process training set
train_shop_dir = os.path.join(output_base, "train", "shoplifting")
train_normal_dir = os.path.join(output_base, "train", "normal")

if os.path.exists(train_shop_dir):
    for frame in os.listdir(train_shop_dir):
        if frame.endswith(('.jpg', '.jpeg', '.png')):
            frame_data.append({
                'frame_path': os.path.join(train_shop_dir, frame),
                'label': 'shoplifting',
                'split': 'train'
            })

if os.path.exists(train_normal_dir):
    for frame in os.listdir(train_normal_dir):
        if frame.endswith(('.jpg', '.jpeg', '.png')):
            frame_data.append({
                'frame_path': os.path.join(train_normal_dir, frame),
                'label': 'normal',
                'split': 'train'
            })

# Create DataFrame
df = pd.DataFrame(frame_data)

print("DataFrame Summary:")
print("-----------------")
print(f"Total frames: {len(df)}")
print("\nLabel distribution:")
print(df['label'].value_counts())
print("\nSplit distribution:")
print(df['split'].value_counts())
print("\nFirst few rows:")
print(df.head())

DataFrame Summary:
-----------------
Total frames: 12510

Label distribution:
label
normal         10290
shoplifting     2220
Name: count, dtype: int64

Split distribution:
split
train    12510
Name: count, dtype: int64

First few rows:
                                          frame_path        label  split
0  processed_frames\train\shoplifting\shoplifting...  shoplifting  train
1  processed_frames\train\shoplifting\shoplifting...  shoplifting  train
2  processed_frames\train\shoplifting\shoplifting...  shoplifting  train
3  processed_frames\train\shoplifting\shoplifting...  shoplifting  train
4  processed_frames\train\shoplifting\shoplifting...  shoplifting  train


In [26]:
def extract_features(frame_dir, batch_size=32):
    """
    Extract features from frames using VGG16.
    
    Args:
        frame_dir (str): Directory containing frames
        batch_size (int): Batch size for processing
    
    Returns:
        features (np.array): Extracted features
        labels (np.array): Corresponding labels
    """
    # Load VGG16 model without top layers
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # Get list of frames
    frames = []
    labels = []
    for class_name in os.listdir(frame_dir):
        class_dir = os.path.join(frame_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
            
        for frame_name in os.listdir(class_dir):
            frames.append(os.path.join(class_dir, frame_name))
            labels.append(1 if class_name == "shoplifting" else 0)
    
    # Convert labels to numpy array
    labels = np.array(labels)
    
    # Process frames in batches
    features = []
    for i in tqdm(range(0, len(frames), batch_size), desc="Extracting features"):
        batch_frames = frames[i:i + batch_size]
        
        # Load and preprocess batch
        batch_data = []
        for frame_path in batch_frames:
            img = load_img(frame_path, target_size=(224, 224))
            x = img_to_array(img)
            x = preprocess_input(x)
            batch_data.append(x)
        
        batch_data = np.array(batch_data)
        
        # Extract features
        batch_features = base_model.predict(batch_data)
        batch_features = batch_features.reshape(batch_features.shape[0], -1)
        features.extend(batch_features)
    
    features = np.array(features)
    
    # Scale features
    scaler = StandardScaler()
    features = scaler.fit_transform(features)
    
    return features, labels

In [31]:
# Create DataFrame in the tutorial's format with error checking
video_paths = []
targets = []

# First, verify the base directory exists
base_dir = "extracted_folder/DCSASS"
if not os.path.exists(base_dir):
    print(f"Error: Directory '{base_dir}' not found!")
else:
    # Get all categories and sort them for consistent processing
    categories = sorted(os.listdir(base_dir))
    print(f"Found {len(categories)} categories: {categories}")
    
    # First, collect all videos and their labels
    for category in categories:
        category_path = os.path.join(base_dir, category)
        if os.path.isdir(category_path):
            print(f"\nProcessing category: {category}")
            video_count = 0
            
            for video_dir in os.listdir(category_path):
                video_dir_path = os.path.join(category_path, video_dir)
                if os.path.isdir(video_dir_path):
                    # Convert to tutorial format path
                    formatted_path = f"data/Shoplifting/{os.path.basename(video_dir)}"
                    if not formatted_path.endswith('.mp4'):
                        formatted_path += '.mp4'
                    video_paths.append(formatted_path)
                    
                    # Set target (1 for shoplifting, 0 for normal)
                    is_shoplifting = 1 if category.lower() == "shoplifting" else 0
                    targets.append(is_shoplifting)
                    video_count += 1
            
            print(f"Added {video_count} videos from {category}")

    # Create DataFrame with the exact same structure as tutorial
    if video_paths and targets:
        df = pd.DataFrame({
            'path': video_paths,
            'target': targets
        })
        
        print("\nDataFrame created successfully:")
        print("-" * 40)
        print(f"Total videos: {len(df)}")
        print(f"Target distribution:\n{df['target'].value_counts()}")
        print("\nFirst few entries:")
        print(df.head())
    else:
        print("Error: No videos found to create DataFrame!")

Found 14 categories: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'Labels', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']

Processing category: Abuse
Added 39 videos from Abuse

Processing category: Arrest
Added 26 videos from Arrest

Processing category: Arson
Added 22 videos from Arson

Processing category: Assault
Added 23 videos from Assault

Processing category: Burglary
Added 47 videos from Burglary

Processing category: Explosion
Added 23 videos from Explosion

Processing category: Fighting
Added 8 videos from Fighting

Processing category: Labels
Added 0 videos from Labels

Processing category: RoadAccidents
Added 76 videos from RoadAccidents

Processing category: Robbery
Added 105 videos from Robbery

Processing category: Shooting
Added 30 videos from Shooting

Processing category: Shoplifting
Added 28 videos from Shoplifting

Processing category: Stealing
Added 64 videos from Stealing

Processing category: Vand